In [1]:
import json
import numpy as np
from pathlib import Path
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" 
import torch
from PIL import Image
from tqdm import tqdm
from pycocotools import mask as mask_utils
from transformers import Sam3Processor, Sam3Model

In [2]:
DATA_PATH = Path("/kaggle/input/datasets/hycloud/sa-1b-part-000999/SA-1B-Part-000999")
OUTPUT_DIR = Path("/kaggle/working/experiments/2026-05-15_visual-degradation-task")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sigmas_list = [0, 10, 25, 50]
file_paths = {sigma: OUTPUT_DIR / f"visual_degradation_sigma{sigma}.jsonl" for sigma in sigmas_list}

In [3]:
def process_results(results):
    masks = results['masks'].cpu().numpy()
    scores = results['scores'].cpu().numpy()
    boxes = results['boxes'].cpu().numpy()

    instance_preds = []
    for i in range(len(masks)):
        mask_binary = (masks[i] > 0).astype(np.uint8)
        rle = mask_utils.encode(np.asfortranarray(mask_binary))
        rle['counts'] = rle['counts'].decode('utf-8')
        instance_preds.append({
            "segmentation": rle,
            "score": float(scores[i]),
            "bbox": boxes[i].tolist()
        })
    return instance_preds


def add_awgn(image: np.ndarray, sigma: float) -> np.ndarray:
    noise = np.random.normal(0, sigma, image.shape)
    noisy = image + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

In [4]:
from huggingface_hub import login
login()

In [5]:
print("Loading SAM3 model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Sam3Model.from_pretrained("facebook/sam3")
processor = Sam3Processor.from_pretrained("facebook/sam3")

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)
print(f"Using {device}...")

model.to(device, dtype=torch.float16)


Loading SAM3 model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

Using 2 GPUs!
Using cuda...


DataParallel(
  (module): Sam3Model(
    (vision_encoder): Sam3VisionModel(
      (backbone): Sam3ViTModel(
        (embeddings): Sam3ViTEmbeddings(
          (patch_embeddings): Sam3ViTPatchEmbeddings(
            (projection): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          )
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (layer_norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (layers): ModuleList(
          (0-31): 32 x Sam3ViTLayer(
            (layer_norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (rotary_emb): Sam3ViTRotaryEmbedding()
            (attention): Sam3ViTRoPEAttention(
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (o_proj): Linear(in_features=1024, out_features=1024, bi

In [7]:
with open("/kaggle/input/datasets/yaramahrous/tempdata/temp.jsonl", "r", encoding="utf-8") as f:
    strings_to_remove = {line.strip() for line in f if line.strip()}


In [8]:
file_stems = sorted(p.stem for p in DATA_PATH.glob("*.jpg"))
file_stems = [item for item in file_stems if item not in strings_to_remove]
print(f"Found {len(file_stems)} images in {DATA_PATH}\n")

Found 4502 images in /kaggle/input/datasets/hycloud/sa-1b-part-000999/SA-1B-Part-000999



In [9]:
from torch.utils.data import Dataset, DataLoader

MAX_SIZE = 512

class ImageDataset(Dataset):
    def __init__(self, tasks, data_path):
        self.tasks = tasks
        self.data_path = data_path

    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        stem, sigma = self.tasks[idx]
        img_path = self.data_path / f"{stem}.jpg"
        img = Image.open(img_path).convert('RGB')
        # img.thumbnail((MAX_SIZE, MAX_SIZE), Image.LANCZOS) 
        img_arr = np.array(img)
        
        if sigma > 0:
            img_arr = add_awgn(img_arr, sigma)
            
        return {"image": img_arr, "stem": stem, "sigma": sigma}
        
def custom_collate(batch):
    """Keep images as a list since they may have different sizes."""
    images = [item['image'] for item in batch]
    stems = [item['stem'] for item in batch]
    sigmas = torch.tensor([item['sigma'] for item in batch])
    return {'image': images, 'stem': stems, 'sigma': sigmas}


tasks = [(stem, sigma) for stem in file_stems for sigma in sigmas_list]
BATCH_SIZE = 2
dataset = ImageDataset(tasks, DATA_PATH)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,collate_fn=custom_collate)

In [10]:
SHARED_PROMPT = "Visual"

In [ ]:
for sigma, path in file_paths.items():
    open(path, "w").close() 
file_handles = {sigma: open(path, "w", encoding="utf-8") for sigma, path in file_paths.items()}
try:
    for batch in tqdm(loader, desc="Processing"):
        torch.cuda.empty_cache()
    
        imgs = batch['image']  
        stems = batch['stem']
        sigmas = batch['sigma'].tolist()
        
        inputs = processor(
            images=imgs, 
            text=[SHARED_PROMPT] * len(imgs), 
            return_tensors="pt"
        ).to(device,dtype=torch.float16)
    
        with torch.no_grad():
            outputs = model(**inputs)
    
        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=0.5,
            mask_threshold=0.5,
            target_sizes=inputs["original_sizes"].cpu().tolist()        
        )
    
        for idx, single_result in enumerate(results):
                sigma_val = sigmas[idx]
                entry = {
                    "image_name": stems[idx],
                    "sigma": sigma_val,
                    "predictions": process_results(single_result)
                }
                file_handles[sigma_val].write(json.dumps(entry) + "\n")
finally:
    for fh in file_handles.values():
        fh.close()

Processing:  74%|███████▍  | 6684/9004 [3:54:07<1:09:17,  1.79s/it]

In [ ]:
import os
import zipfile
from IPython.display import FileLink

def zip_and_download(folder_path, output_zip_name):
    """
    Zips a specified folder and creates a clickable download link in Kaggle.
    """
    # Append .zip if it's not there
    if not output_zip_name.endswith('.zip'):
        output_zip_name += '.zip'
        
    # Create the zip file
    with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                # Create a relative path to keep the folder structure clean
                file_path = os.path.join(root, file)
                archive_name = os.path.relpath(file_path, start=folder_path)
                zipf.write(file_path, archive_name)
                
    print(f"Folder zipped successfully as: {output_zip_name}")
    
    # Generate and return the download link
    return FileLink(output_zip_name)

# --- How to use it ---
# Kaggle's default output directory is '/kaggle/working'
# If you want to download everything in the output, use '.' (current directory)
zip_and_download('/kaggle/working/experiments/2026-05-15_visual-degradation-task', 'my_kaggle_output.zip')